In [2]:
import re

def script_consistency_check(text, min_arabic_ratio=0.5, max_latin_ratio=0.25):
    """Flag mixed-script contamination. Two triggers:
    1. Global: too high a share of Latin/Cyrillic letters overall.
    2. Token-level: standalone Latin-letter tokens inside Arabic-dominant text
       (catches diluted contamination like sample 14 that averaging misses)."""
    if not text or not text.strip():
        return True, "empty output"

    arabic = len(re.findall(r'[\u0600-\u06FF]', text))
    latin_cyrillic = len(re.findall(r'[A-Za-z\u0400-\u04FF]', text))
    total_letters = arabic + latin_cyrillic
    if total_letters == 0:
        return True, "no recognizable letters"

    # Trigger 1: global ratio (catches heavy contamination)
    latin_ratio = latin_cyrillic / total_letters
    if latin_ratio > max_latin_ratio:
        return True, f"mixed-script: {latin_ratio:.0%} non-Arabic letters"

    # Trigger 2: standalone Latin tokens in Arabic-dominant text
    # (catches diluted contamination — Latin fragments amid correct Arabic)
    is_arabic_dominant = (arabic / total_letters) >= min_arabic_ratio
    if is_arabic_dominant:
        tokens = text.split()
        latin_tokens = [t for t in tokens if re.search(r'[A-Za-z]', t)]
        if latin_tokens:
            return True, f"Latin fragments in Arabic text: {latin_tokens[:3]}"

    return False, "script consistent"


def repetition_check(text, max_repeat_ratio=0.5):
    words = text.split()
    if len(words) < 4:
        return False, "too short to assess repetition"
    most_common_count = max(words.count(w) for w in set(words))
    repeat_ratio = most_common_count / len(words)
    if repeat_ratio > max_repeat_ratio:
        return True, f"repetitive: one token is {repeat_ratio:.0%} of output"
    return False, "no excessive repetition"


test_cases = [
    ("Sample 43 (repetition)",  "كالمراه المقالت عالم الله المجادة عالم الا فالمتالات المالية المالية المالية المالية المالية المالية المالية المالية المالية المالية المالية المالية"),
    ("Sample 14 (Latin contam)", "ws 10119 w 301 w111 تتحمل المسؤولية المتعلقة بخطة التحريري. وفضلا عن ذلك"),
    ("Sample 13 (subtle wrong)", "وَإِذا ما سَأَلَتْنِي عَن مَعْنَى لَفَظَهُ \"عرب\" عِنْدَ"),
    ("Sample 2 (clean)",         "تُثِير فِيها الاسود، وَهُوَ ما يَتَعارَض مَعَ اِكْتِشافِ"),
    ("Legit mixed (COVID-19 ref)", "تنتشر جائحة كوفيد في جميع أنحاء العالم بسرعة كبيرة جدا"),  # control: should PASS
]

print(f"{'Case':<30}{'Script':<10}{'Repeat':<10}{'Reason'}")
print("-" * 85)
for name, text in test_cases:
    s_flag, s_reason = script_consistency_check(text)
    r_flag, r_reason = repetition_check(text)
    reason = s_reason if s_flag else (r_reason if r_flag else "clean")
    print(f"{name:<30}{('FLAG' if s_flag else 'pass'):<10}{('FLAG' if r_flag else 'pass'):<10}{reason}")

Case                          Script    Repeat    Reason
-------------------------------------------------------------------------------------
Sample 43 (repetition)        pass      FLAG      repetitive: one token is 60% of output
Sample 14 (Latin contam)      FLAG      pass      Latin fragments in Arabic text: ['ws', 'w', 'w111']
Sample 13 (subtle wrong)      pass      pass      clean
Sample 2 (clean)              pass      pass      clean
Legit mixed (COVID-19 ref)    pass      pass      clean
